In [1]:
!pip install -U langchain langchain-ollama langchain-core

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import time

c:\Users\amjat\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model  = ChatOllama(model='mistral', temperature=0.3)
parser = StrOutputParser() #gives Plain string

In [4]:
outline_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are a professional content planner. Be concise.'),
        ('human',  'Create a 5-point outline for an article about: {topic}')
    ]) | model | parser
)

In [5]:
expand_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are a professional writer. Write clearly and concisely.'),
        ('human',  'Expand this outline into a full article (300 words max):\n\n{outline}')
    ]) | model | parser
)

In [6]:
sequential_chain = (
    {
        'outline': outline_chain,   # Step 1 output
        'topic': RunnablePassthrough()  # original input
    } | expand_chain
)

In [9]:
topic = 'How Indian street food vendors can use UPI to grow their business'

print(f'Writing article about: {topic}\n')
print('─' * 60)

t0 = time.time()
token_count = 0

for chunk in sequential_chain.stream({'topic': topic}):
    print(chunk, end='', flush=True)
    token_count += len(chunk.split())

elapsed = time.time() - t0

print('\n' + '─' * 60)
print(f'Done in {elapsed:.1f}s | ~{token_count} words generated')

Writing article about: How Indian street food vendors can use UPI to grow their business

────────────────────────────────────────────────────────────
 Title: Empowering Indian Street Food Vendors with UPI: A Pathway to Business Growth

The vibrant Indian street food industry, a cultural cornerstone and economic powerhouse, faces challenges in the digital payments arena. Traditional cash transactions pose risks of theft and mismanagement, hindering financial growth for vendors. Enter UPI (Unified Payments Interface), a potential game-changer.

UPI, a real-time payment system developed by the National Payments Corporation of India, offers a seamless solution to these issues. By linking multiple bank accounts into a single mobile application, UPI enables instant fund transfers between any two parties with just a virtual ID.

For street food vendors, UPI promises ease of transactions, reduced cash handling, improved financial records, and increased customer trust. The benefits are tangibl